# `Industrial Machine Learning on Hadoop and Spark`
## `Seminar 06: Spark DataFrame Basics`

### `Maks Nakhodnov`
#### `Bremen, 2025`

From this notebook, you can learn about:

* DataFrame and SQL API — how to create, query, and manipulate structured data using Spark.
* Basic operations in Spark — fundamental transformations and actions you can perform on RDDs and DataFrames.

## `Initialization`

In [1]:
! pip3 install pyspark pyarrow kaggle parquet-tools

In [1]:
import os
import sys

os.environ['PYSPARK_DRIVER_PYTHON'] = os.environ['PYSPARK_PYTHON'] = sys.executable
! rm -rf ./m5-forecasting-accuracy ./base_statistics.csv/ ./base_statistics.parquet/

In [2]:
from pyspark.sql import SparkSession
from pyspark import SparkConf, SparkContext

# Create a configuration class with connection parameters to the server
conf = (
    SparkConf()
        # Specify the port for the UI
        .set('spark.ui.port', '4050')
        # Specify the URL of the master node of the Spark cluster
        # You can use local mode by specifying `local[<number_cores>]`
        # In this case, all processing will occur on the current machine
        # This can provide an advantage due to parallelism across CPU cores
        .setMaster('local[*]')
        # If you need to connect to a "real" cluster, you should specify the URL `spark://<master-node-url:master-node-port>`. For example:
        # .setMaster('spark://localhost:7077')
)
# Create an access point to the cluster. Allows using RDD API
sc = SparkContext(conf=conf)
# Access point for using the DataFrame API
spark = SparkSession(sc)

# At the end of the program, make sure to stop the connection to free up resources
# sc.stop()

25/10/20 22:47:03 WARN Utils: Your hostname, 2A2I-PRO-3SMD6R.local resolves to a loopback address: 127.0.0.1; using 192.168.1.47 instead (on interface en0)
25/10/20 22:47:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/20 22:47:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## `RDD API`

In [3]:
import glob

In [6]:
data = [1, 2, 3, 4, 5]
rdd = spark.sparkContext.parallelize(data).repartition(2)
rdd, rdd.collect()

(MapPartitionsRDD[11] at coalesce at NativeMethodAccessorImpl.java:0,
 [1, 2, 4, 5, 3])

25/10/20 22:47:16 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [7]:
squared_rdd = rdd.map(lambda x: x * x)
squared_rdd, squared_rdd.collect()

(PythonRDD[12] at collect at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_12063/4065467666.py:2,
 [1, 4, 16, 25, 9])

In [8]:
print(squared_rdd.toDebugString().decode())

(2) PythonRDD[12] at collect at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_12063/4065467666.py:2 []
 |  MapPartitionsRDD[11] at coalesce at NativeMethodAccessorImpl.java:0 []
 |  CoalescedRDD[10] at coalesce at NativeMethodAccessorImpl.java:0 []
 |  ShuffledRDD[9] at coalesce at NativeMethodAccessorImpl.java:0 []
 +-(12) MapPartitionsRDD[8] at coalesce at NativeMethodAccessorImpl.java:0 []
    |   PythonRDD[7] at RDD at PythonRDD.scala:53 []
    |   ParallelCollectionRDD[6] at readRDDFromFile at PythonRDD.scala:289 []


In [9]:
filtered_rdd = rdd.filter(lambda x: x % 2 == 0)
filtered_rdd, filtered_rdd.count(), filtered_rdd.take(1)

(PythonRDD[15] at RDD at PythonRDD.scala:53, 2, [2])

In [10]:
sum_of_elements = rdd.reduce(lambda a, b: a + b)
sum_of_elements

15

In [11]:
rdd.map(lambda x: (x % 2, x)).reduceByKey(lambda a, b: a + b).collect()

[(0, 6), (1, 9)]

In [12]:
filtered_rdd.union(squared_rdd).collect()

[2, 4, 1, 4, 16, 25, 9]

In [13]:
squared_rdd.sample(withReplacement=True, fraction=0.5).collect()

[1, 4, 16, 9]

In [14]:
! rm -rf ./rdd.txt/
rdd.saveAsTextFile('./rdd.txt')

rdd_file = glob.glob('./rdd.txt/part-*0')[0]
! cat $rdd_file
rdd_file = glob.glob('./rdd.txt/part-*1')[0]
! cat $rdd_file

1
2
4
5
3


In [15]:
visits = [
    ('index.html', '1.2.3.4'),
    ('about.html', '3.4.5.6'),
    ('index.html', '1.3.3.1'),
]
visits = spark.sparkContext.parallelize(visits)

pagenames = [
    ('index.html', 'Home'),
    ('about.html', 'About'),
]
pagenames = spark.sparkContext.parallelize(pagenames)

joined_rdd = visits.join(pagenames)
joined_rdd.collect()

[('about.html', ('3.4.5.6', 'About')),
 ('index.html', ('1.2.3.4', 'Home')),
 ('index.html', ('1.3.3.1', 'Home'))]

In [16]:
joined_rdd = visits.cogroup(pagenames)
joined_rdd.mapValues(lambda x: [list(_) for _ in x]).collect()

[('about.html', [['3.4.5.6'], ['About']]),
 ('index.html', [['1.2.3.4', '1.3.3.1'], ['Home']])]

#### `WordCount RDD`

In [17]:
text_path = './../../../docker-hadoop-spark/examples/wordcount_streaming/data/input/input.01.txt'

In [18]:
! head $text_path

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Morbi sodales, metus in luctus varius, arcu nunc aliquet lorem, sit amet pulvinar odio ex nec dui. Suspendisse vitae feugiat diam. Vivamus finibus sit amet turpis vitae tristique. Orci varius natoque penatibus et magnis dis parturient montes, nascetur ridiculus mus. Praesent tempor efficitur velit, et efficitur ligula tempor sed. Integer tincidunt in ante eget semper. Nunc a sem quis metus eleifend bibendum.

Mauris id leo elit. Aenean urna lectus, condimentum at mollis vitae, commodo eu nibh. Morbi at ornare felis, at pretium felis. Fusce in quam eget purus mattis hendrerit. Nunc hendrerit turpis vel tellus malesuada accumsan. Nulla auctor nulla vel tempor dignissim. Suspendisse sagittis leo risus.

Nam consectetur fermentum eleifend. Aenean a erat vitae magna rutrum placerat. Ut eget diam bibendum mauris porttitor gravida. Morbi pulvinar ante finibus, interdum turpis et, posuere ligula. Nunc sodales sed diam id pharetra. Proin 

In [19]:
from pyspark import StorageLevel

In [20]:
file = spark.sparkContext.textFile(text_path)

part_01 = (
    file
        .flatMap(lambda line: line.split(' '))
        .map(lambda word: (word, 1))
        .persist(StorageLevel.DISK_ONLY)
)
counts = (
    part_01
        .reduceByKey(lambda a, b: a + b)
        .sortBy(lambda x: -x[1])
)
counts.take(10)

[('et', 197),
 ('quis', 169),
 ('ac', 160),
 ('sit', 159),
 ('vitae', 153),
 ('in', 152),
 ('', 149),
 ('vel', 149),
 ('ut', 148),
 ('id', 143)]

In [21]:
print(part_01.toDebugString().decode())

(2) PythonRDD[47] at RDD at PythonRDD.scala:53 [Disk Serialized 1x Replicated]
 |       CachedPartitions: 2; MemorySize: 0.0 B; DiskSize: 51.4 KiB
 |  ./../../../docker-hadoop-spark/examples/wordcount_streaming/data/input/input.01.txt MapPartitionsRDD[46] at textFile at NativeMethodAccessorImpl.java:0 [Disk Serialized 1x Replicated]
 |  ./../../../docker-hadoop-spark/examples/wordcount_streaming/data/input/input.01.txt HadoopRDD[45] at textFile at NativeMethodAccessorImpl.java:0 [Disk Serialized 1x Replicated]


In [22]:
print(counts.toDebugString().decode())

(2) PythonRDD[59] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[57] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[56] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[55] at sortBy at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_12063/1069691849.py:12 []
    |  PythonRDD[54] at sortBy at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_12063/1069691849.py:12 []
    |  MapPartitionsRDD[51] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[50] at partitionBy at NativeMethodAccessorImpl.java:0 []
    +-(2) PairwiseRDD[49] at reduceByKey at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_12063/1069691849.py:11 []
       |  PythonRDD[48] at reduceByKey at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_12063/1069691849.py:11 []
       |  PythonRDD[47] at RDD at PythonRDD.scala:53 []
       |      CachedPartitions: 2; MemorySize: 0.0 B; DiskSize: 51.4 KiB
       |  ./../../../docker-hadoop-sp

## `DataFrame API`

### `Data loading`

First, you need to download `kaggle.json` from your [Kaggle account settings](https://www.kaggle.com/settings). Place it in the folder `~/.kaggle`. On Linux/MacOS, this can be done as follows:

In [23]:
! mkdir ~/.kaggle/
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json

mkdir: /Users/nakhodnov/.kaggle/: File exists
cp: kaggle.json: No such file or directory


In this project, you will work with the data for demand forecasting: [M5 Forecasting](https://www.kaggle.com/competitions/m5-forecasting-accuracy/data).

In [24]:
path = './m5-forecasting-accuracy'

In [25]:
! kaggle competitions download -c m5-forecasting-accuracy

import zipfile
with zipfile.ZipFile('./m5-forecasting-accuracy.zip', 'r') as zip_ref:
    zip_ref.extractall(path)

m5-forecasting-accuracy.zip: Skipping, found more recently modified local copy (use --force to force download)


In [26]:
%ls $path

calendar.csv                sample_submission.csv
sales_train_evaluation.csv  sell_prices.csv
sales_train_validation.csv


In [27]:
# Set paths to files from the dataset
file_calendar = f"{path}/calendar.csv"
file_validation = f"{path}/sales_train_validation.csv"
file_evaluation = f"{path}/sales_train_evaluation.csv"
file_prices = f"{path}/sell_prices.csv"

# Data format — CSV
file_type = "csv"
# Set options for how to interpret loaded data
# Automatically determine column types
infer_schema = "true"
# Interpret the first row in the file as column names
first_row_is_header = "true"
# Set the delimiter between column values
delimiter = ","

df_validation = (
    spark.read.format(file_type)
      .option("inferSchema", infer_schema)
      .option("header", first_row_is_header)
      .option("sep", delimiter)
      .load(file_validation)
    # Load is neither action or transformation
    # You can also specify paths in HDFS or databases, e.g., Hive
#       .load('hdfs:///path_to_data/...')
)

df_evaluation = (
    spark.read.format(file_type)
      .option("inferSchema", infer_schema)
      .option("header", first_row_is_header)
      .option("sep", delimiter)
      .load(file_evaluation)
)
df_prices = (
    spark.read.format(file_type)
      .option("inferSchema", infer_schema)
      .option("header", first_row_is_header)
      .option("sep", delimiter)
      .load(file_prices)
)

# Take the first 10 rows of the pyspark.sql.dataframe.DataFrame
# And perform an action to convert it into a pandas.DataFrame
df_validation.limit(10).toPandas()

25/10/20 22:48:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4
5,HOBBIES_1_006_CA_1_validation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,0,1,0,0,0,2,0,0
6,HOBBIES_1_007_CA_1_validation,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,1,0,1,0,0,1,1
7,HOBBIES_1_008_CA_1_validation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,12,15,0,0,...,0,0,1,37,3,4,6,3,2,1
8,HOBBIES_1_009_CA_1_validation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,2,0,7,3,...,0,0,1,1,6,0,0,0,0,0
9,HOBBIES_1_010_CA_1_validation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,0,0,1,0,...,1,0,0,0,0,0,0,2,0,2


### `Spark DataFrame API`

* [Quickstart](https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html)
* [Docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)

In [28]:
emp_data = [
    (1, 'Smith', 10),
    (2, 'Rose', 20),
    (3, 'Williams', 10),
    (4, 'Jones', 30),
    (5, 'Jones', None),
]
emp_columns = ['emp_id', 'name', 'dept_id']

emp_df = spark.createDataFrame(emp_data, emp_columns)
emp_df

DataFrame[emp_id: bigint, name: string, dept_id: bigint]

In [29]:
type(emp_df)

pyspark.sql.dataframe.DataFrame

The DataFrame output does not show its contents because it has not been computed yet; in Spark, computations are only executed when an action is called.

Examples of actions:

* `count()` — counts the number of rows in the DataFrame
* `toPandas()` — converts a Spark DataFrame into a pandas DataFrame
* `collect()` — performs the computation of the current Spark DataFrame and returns the result
* `show()` — `collect()` + pretty print of the result

In [30]:
emp_df.show()

+------+--------+-------+
|emp_id|    name|dept_id|
+------+--------+-------+
|     1|   Smith|     10|
|     2|    Rose|     20|
|     3|Williams|     10|
|     4|   Jones|     30|
|     5|   Jones|   NULL|
+------+--------+-------+



Basic information about the data — column names and their types:

In [31]:
emp_df.columns, emp_df.schema

(['emp_id', 'name', 'dept_id'],
 StructType([StructField('emp_id', LongType(), True), StructField('name', StringType(), True), StructField('dept_id', LongType(), True)]))

Many methods are similar to those in `pandas.DataFrame`:

In [32]:
emp_df.dropna().show()

+------+--------+-------+
|emp_id|    name|dept_id|
+------+--------+-------+
|     1|   Smith|     10|
|     2|    Rose|     20|
|     3|Williams|     10|
|     4|   Jones|     30|
+------+--------+-------+



A DataFrame consists of columns. You can access a column via attributes or indexing:

In [33]:
emp_df.name, emp_df['name']

(Column<'name'>, Column<'name'>)

Instead of the actual column values, in line with Spark’s “lazy” computation principle, references to them are returned. These columns can participate in symbolic computations. For example, arithmetic or boolean operations can be applied to them.

In [34]:
column_expr = (emp_df.dept_id - 20) / 10 > emp_df.emp_id
column_expr

Column<'(((dept_id - 20) / 10) > emp_id)'>

The resulting **column expressions** can then be computed:

In [35]:
emp_df.select(column_expr).show()

+--------------------------------+
|(((dept_id - 20) / 10) > emp_id)|
+--------------------------------+
|                           false|
|                           false|
|                           false|
|                           false|
|                            NULL|
+--------------------------------+



You can also rename a column:

In [36]:
emp_df.select((emp_df.dept_id ** 2).alias('dept_id squared')).show()

+---------------+
|dept_id squared|
+---------------+
|          100.0|
|          400.0|
|          100.0|
|          900.0|
|           NULL|
+---------------+



SQL-like operations are available on DataFrames, for example, `join`:

In [37]:
dept_data = [
    ('Finance', 10),
    ('Marketing', 20),
    ('Sales', 30),
    ('IT', 40),
]
dept_columns = ['dept_name', 'dept_id']

dept_df = spark.createDataFrame(dept_data, dept_columns)
dept_df.show()

+---------+-------+
|dept_name|dept_id|
+---------+-------+
|  Finance|     10|
|Marketing|     20|
|    Sales|     30|
|       IT|     40|
+---------+-------+



In [38]:
emp_df.join(dept_df, how='inner', on=['dept_id']).show()

+-------+------+--------+---------+
|dept_id|emp_id|    name|dept_name|
+-------+------+--------+---------+
|     10|     1|   Smith|  Finance|
|     10|     3|Williams|  Finance|
|     20|     2|    Rose|Marketing|
|     30|     4|   Jones|    Sales|
+-------+------+--------+---------+



In [39]:
emp_df.join(dept_df, how='outer', on=['dept_id']).show()

+-------+------+--------+---------+
|dept_id|emp_id|    name|dept_name|
+-------+------+--------+---------+
|   NULL|     5|   Jones|     NULL|
|     10|     3|Williams|  Finance|
|     10|     1|   Smith|  Finance|
|     20|     2|    Rose|Marketing|
|     30|     4|   Jones|    Sales|
|     40|  NULL|    NULL|       IT|
+-------+------+--------+---------+



Filtering and sorting are also supported:

In [40]:
(emp_df['name'] == 'Smith') | (emp_df['name'] == 'Rose')

Column<'((name = Smith) OR (name = Rose))'>

In [41]:
(
    emp_df
      .join(dept_df, how='outer', on=['dept_id'])
      # Note the use of column expressions in the filter
      .where((emp_df['name'] == 'Smith') | (emp_df['name'] == 'Rose'))
      .sort('dept_id')
      .show()
)

+-------+------+-----+---------+
|dept_id|emp_id| name|dept_name|
+-------+------+-----+---------+
|     10|     1|Smith|  Finance|
|     20|     2| Rose|Marketing|
+-------+------+-----+---------+



Working with columns is usually done via column expressions. They can also be used for joins:

In [42]:
emp_columns_renamed = ['emp_id', 'name', 'emp_dept_id']

emp_renamed_df = spark.createDataFrame(emp_data, emp_columns_renamed)
emp_renamed_df.show()

+------+--------+-----------+
|emp_id|    name|emp_dept_id|
+------+--------+-----------+
|     1|   Smith|         10|
|     2|    Rose|         20|
|     3|Williams|         10|
|     4|   Jones|         30|
|     5|   Jones|       NULL|
+------+--------+-----------+



In [43]:
emp_renamed_df.join(
    dept_df, emp_renamed_df.emp_dept_id == dept_df.dept_id,  how='inner'
).show()

+------+--------+-----------+---------+-------+
|emp_id|    name|emp_dept_id|dept_name|dept_id|
+------+--------+-----------+---------+-------+
|     1|   Smith|         10|  Finance|     10|
|     3|Williams|         10|  Finance|     10|
|     2|    Rose|         20|Marketing|     20|
|     4|   Jones|         30|    Sales|     30|
+------+--------+-----------+---------+-------+



Column renaming is also possible:

In [44]:
(
    emp_renamed_df
      .withColumnRenamed('emp_dept_id', 'dept_id')
      .join(
          dept_df, 'dept_id', how='inner'
      )
      .show()
)

+-------+------+--------+---------+
|dept_id|emp_id|    name|dept_name|
+-------+------+--------+---------+
|     10|     1|   Smith|  Finance|
|     10|     3|Williams|  Finance|
|     20|     2|    Rose|Marketing|
|     30|     4|   Jones|    Sales|
+-------+------+--------+---------+



The module `pyspark.sql.functions` contains a large set of helper functions for transforming columns. For example:

* Helper functions: `lit`, `col`, ...
* Element-wise mathematical functions: `cos`, `sin`, `round`, ...
* Element-wise date and time functions: `dayofmonth`, ...
* Aggregators: `sum`, `mean`, ...
* Functions for working with complex data in columns: `array_sort`, `concat`, ...
* Sorting: `asc`, ...
* String functions: `concat_ws`, `lower`, `split`, ...
* Window functions: `lag`, ...
* Transformations with user-defined functions: `udf_pandas`, ...

**Always check if a ready-made function exists in this module before writing code. Using built-in functions significantly affects computation speed.**

In [45]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType

Often, to apply functions, you need to change the column type:

In [46]:
emp_with_date = (
    emp_df
        .dropna()
        .withColumn(
            'hire_date', 
            # Construct a date in yyyy-mm-dd format
            F.concat_ws(
                '-', 
                # Generate year
                (1990 + emp_df.dept_id).cast(StringType()),
                # Generate month
                F.concat(F.lit('0'), emp_df.emp_id.cast(StringType())), 
                # Generate day
                emp_df.dept_id.cast(StringType())
            ).cast(DateType())
        )
)
emp_with_date.show()

+------+--------+-------+----------+
|emp_id|    name|dept_id| hire_date|
+------+--------+-------+----------+
|     1|   Smith|     10|2000-01-10|
|     2|    Rose|     20|2010-02-20|
|     3|Williams|     10|2000-03-10|
|     4|   Jones|     30|2020-04-30|
+------+--------+-------+----------+



In [47]:
emp_with_date.select(
    F.acos(emp_with_date.emp_id / 4),
    F.year(emp_with_date.hire_date),
    F.regexp_replace(F.lower(emp_with_date.name), 'smith', '史密斯').alias('processed_name')
).show()

+------------------+---------------+--------------+
|ACOS((emp_id / 4))|year(hire_date)|processed_name|
+------------------+---------------+--------------+
| 1.318116071652818|           2000|        史密斯|
|1.0471975511965979|           2010|          rose|
|0.7227342478134157|           2000|      williams|
|               0.0|           2020|         jones|
+------------------+---------------+--------------+



### `Spark SQL API`

In [48]:
add_data = [
    (1, '1523 Main St', 'SFO', 'CA'),
    (2, '3453 Orange St', 'SFO', 'NY'),
    (3, '34 Warner St', 'Jersey', 'NJ'),
    (4, '221 Cavalier St', 'Newark', 'DE'),
    (5, '789 Walnut St', 'Sandiago', 'CA')
]
add_columns = ['emp_id', 'address', 'city', 'state']

add_df = spark.createDataFrame(add_data, add_columns)
add_df.show()

+------+---------------+--------+-----+
|emp_id|        address|    city|state|
+------+---------------+--------+-----+
|     1|   1523 Main St|     SFO|   CA|
|     2| 3453 Orange St|     SFO|   NY|
|     3|   34 Warner St|  Jersey|   NJ|
|     4|221 Cavalier St|  Newark|   DE|
|     5|  789 Walnut St|Sandiago|   CA|
+------+---------------+--------+-----+



Spark allows you to use a DataFrame as a table in regular SQL queries:

In [49]:
emp_df.createOrReplaceTempView('EMP')
dept_df.createOrReplaceTempView('DEPT')
add_df.createOrReplaceTempView('ADD')

In [50]:
spark.sql('''
    select * from EMP e, DEPT d, ADD a
    where e.dept_id == d.dept_id and e.emp_id == a.emp_id
''').show()

+------+--------+-------+---------+-------+------+---------------+------+-----+
|emp_id|    name|dept_id|dept_name|dept_id|emp_id|        address|  city|state|
+------+--------+-------+---------+-------+------+---------------+------+-----+
|     1|   Smith|     10|  Finance|     10|     1|   1523 Main St|   SFO|   CA|
|     2|    Rose|     20|Marketing|     20|     2| 3453 Orange St|   SFO|   NY|
|     3|Williams|     10|  Finance|     10|     3|   34 Warner St|Jersey|   NJ|
|     4|   Jones|     30|    Sales|     30|     4|221 Cavalier St|Newark|   DE|
+------+--------+-------+---------+-------+------+---------------+------+-----+



### `More basic operations on Spark DataFrame`

In [51]:
data = [
    ('James', 'Sales', 3000),
    ('Michael', 'Sales', 4600),
    ('Robert', 'Sales', 4100),
    ('Maria', 'Finance', 3000),
    ('James', 'Sales', 3000),
    ('Scott', 'Finance', 3300),
    ('Jen', 'Finance', 3900),
    ('Jeff', ' Marketing', 3000),
    ('Kumar', 'Marketing', 2000),
    ('Saif', 'Sales', 4100),
]
columns = ['Name', 'Dept', 'Salary']

df = spark.createDataFrame(data, columns)
df.show()

+-------+----------+------+
|   Name|      Dept|Salary|
+-------+----------+------+
|  James|     Sales|  3000|
|Michael|     Sales|  4600|
| Robert|     Sales|  4100|
|  Maria|   Finance|  3000|
|  James|     Sales|  3000|
|  Scott|   Finance|  3300|
|    Jen|   Finance|  3900|
|   Jeff| Marketing|  3000|
|  Kumar| Marketing|  2000|
|   Saif|     Sales|  4100|
+-------+----------+------+



In [52]:
df.distinct().show()

+-------+----------+------+
|   Name|      Dept|Salary|
+-------+----------+------+
|  James|     Sales|  3000|
|Michael|     Sales|  4600|
| Robert|     Sales|  4100|
|  Maria|   Finance|  3000|
|  Scott|   Finance|  3300|
|    Jen|   Finance|  3900|
|   Jeff| Marketing|  3000|
|  Kumar| Marketing|  2000|
|   Saif|     Sales|  4100|
+-------+----------+------+



In [53]:
df.distinct().count()

9

It is also possible to perform grouping and aggregations:

In [54]:
df.groupBy('Dept').sum().collect()

[Row(Dept='Sales', sum(Salary)=18800),
 Row(Dept='Finance', sum(Salary)=10200),
 Row(Dept=' Marketing', sum(Salary)=3000),
 Row(Dept='Marketing', sum(Salary)=2000)]

### `IO operations`

In [55]:
base_statistics = df.select(
    F.min('Salary').alias('min_salary'),
    F.mean('Salary').alias('mean_salary'),
    F.max('Salary').alias('max_salary')
)
# No computations have occurred yet
base_statistics

DataFrame[min_salary: bigint, mean_salary: double, max_salary: bigint]

In [56]:
base_statistics.write.csv('./base_statistics.csv', header=True)
base_statistics.write.parquet('./base_statistics.parquet')

In [57]:
csv_file = glob.glob('./base_statistics.csv/*.csv')[0]
parket_file = glob.glob('./base_statistics.parquet/*.snappy.parquet')[0]

! cat $csv_file
%ls ./base_statistics.parquet/
! parquet-tools inspect $parket_file

min_salary,mean_salary,max_salary
2000,3400.0,4600
_SUCCESS
part-00000-9627a071-7111-44ae-ac4b-5363f65ca82e-c000.snappy.parquet

############ file meta data ############
created_by: parquet-mr version 1.13.1 (build db4183109d5b734ec5930d870cdae161e408ddba)
num_columns: 3
num_rows: 1
num_row_groups: 1
format_version: 1.0
serialized_size: 789


############ Columns ############
min_salary
mean_salary
max_salary

############ Column(min_salary) ############
name: min_salary
path: min_salary
max_definition_level: 1
max_repetition_level: 0
physical_type: INT64
logical_type: None
converted_type (legacy): NONE
compression: SNAPPY (space_saved: -5%)

############ Column(mean_salary) ############
name: mean_salary
path: mean_salary
max_definition_level: 1
max_repetition_level: 0
physical_type: DOUBLE
logical_type: None
converted_type (legacy): NONE
compression: SNAPPY (space_saved: -5%)

############ Column(max_salary) ############
name: max_salary
path: max_salary
max_definition_level: 1
max_r

In [58]:
loaded_df = (
    spark.read
        .format('csv')
        .option("inferSchema", True)
        .option("header", True)
        .option("sep", ',')
        .load('./base_statistics.csv')
)
loaded_df, loaded_df.show()

+----------+-----------+----------+
|min_salary|mean_salary|max_salary|
+----------+-----------+----------+
|      2000|     3400.0|      4600|
+----------+-----------+----------+



(DataFrame[min_salary: int, mean_salary: double, max_salary: int], None)

#### `WordCount DataFrame`

In [59]:
from pyspark.sql.functions import split, explode, count

text_path = './../../../docker-hadoop-spark/examples/wordcount_streaming/data/input/input.01.txt'
df_text = spark.read.text(text_path)

df_words_array = df_text.withColumn('words', split(df_text['value'], ' '))
df_single_words = df_words_array.withColumn('word', explode(df_words_array['words'])).drop('value', 'words')
df_filtered_words = df_single_words.filter(df_single_words['word'] != '')
word_counts_df = df_filtered_words.groupBy('word').agg(count('word').alias('count'))
final_word_counts = word_counts_df.orderBy(word_counts_df['count'].desc())
final_word_counts.show()

+---------+-----+
|     word|count|
+---------+-----+
|       et|  197|
|     quis|  169|
|       ac|  160|
|      sit|  159|
|    vitae|  153|
|       in|  152|
|      vel|  149|
|       ut|  148|
|       id|  143|
|       at|  140|
|       eu|  138|
|     amet|  134|
|      non|  132|
|     eget|  126|
|      nec|  125|
|      Sed|  123|
|      sed|  123|
|        a|  122|
|    Donec|  100|
|tincidunt|   80|
+---------+-----+
only showing top 20 rows



In [60]:
final_word_counts.explain(extended=True)

== Parsed Logical Plan ==
Sort [count#10162L DESC NULLS LAST], true
+- Aggregate [word#10155], [word#10155, count(word#10155) AS count#10162L]
   +- Filter NOT (word#10155 = )
      +- Project [word#10155]
         +- Project [value#10148, words#10150, word#10155]
            +- Generate explode(words#10150), false, [word#10155]
               +- Project [value#10148, split(value#10148,  , -1) AS words#10150]
                  +- Relation [value#10148] text

== Analyzed Logical Plan ==
word: string, count: bigint
Sort [count#10162L DESC NULLS LAST], true
+- Aggregate [word#10155], [word#10155, count(word#10155) AS count#10162L]
   +- Filter NOT (word#10155 = )
      +- Project [word#10155]
         +- Project [value#10148, words#10150, word#10155]
            +- Generate explode(words#10150), false, [word#10155]
               +- Project [value#10148, split(value#10148,  , -1) AS words#10150]
                  +- Relation [value#10148] text

== Optimized Logical Plan ==
Sort [count#101

## `Accumulators`

In [61]:
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType, StringType

In [62]:
data = [
    ('Alice', 25, 'New York'),
    ('Bob', 30, 'London'),
    ('Charlie', 35, 'New York'),
    ('David', 28, 'Paris'),
    ('Eve', 40, 'London'),
]
columns = ['Name', 'Age', 'City']
df = spark.createDataFrame(data, columns)
df.show()

+-------+---+--------+
|   Name|Age|    City|
+-------+---+--------+
|  Alice| 25|New York|
|    Bob| 30|  London|
|Charlie| 35|New York|
|  David| 28|   Paris|
|    Eve| 40|  London|
+-------+---+--------+



In [63]:
accumulator = spark.sparkContext.accumulator(0)

def count(row):
    global accumulator
    if row.City == "New York":
        accumulator.add(1)

df.rdd.foreach(count)
print(f'Number of people from New York: {accumulator.value}')

Number of people from New York: 2


In [64]:
accumulator = spark.sparkContext.accumulator(0)

@udf(IntegerType())
def count_udf(age):
    global accumulator
    if age <= 30:
        accumulator.add(1)
    # UDF must return a value
    return 1

df.withColumn('dummy_col', count_udf(df['Age'])).collect()
print(f'Number of people Age <= 30: {accumulator.value}')

Number of people Age <= 30: 3


## `Broadcast`

In [65]:
state_mapping = {
    'CA': 'California',
    'NY': 'New York',
    'TX': 'Texas',
    'IL': 'Illinois'
}

data = [
    ('Alice', 'CA'), 
    ('Bob', 'NY'), 
    ('Charlie', 'TX'), 
    ('David', 'IL'), 
    ('Eve', 'FL')
]
columns = ['Name', 'StateCode']
df = spark.createDataFrame(data, columns)
df.show()

+-------+---------+
|   Name|StateCode|
+-------+---------+
|  Alice|       CA|
|    Bob|       NY|
|Charlie|       TX|
|  David|       IL|
|    Eve|       FL|
+-------+---------+



In [66]:
broadcast_states = spark.sparkContext.broadcast(state_mapping)

In [67]:
def get_full_state_name(state_code):
    return broadcast_states.value.get(state_code, "Unknown")

rdd_with_full_state = df.rdd.map(
    lambda row: (
        row["Name"], row["StateCode"], get_full_state_name(row["StateCode"])
    )
)
rdd_with_full_state.collect()

[('Alice', 'CA', 'California'),
 ('Bob', 'NY', 'New York'),
 ('Charlie', 'TX', 'Texas'),
 ('David', 'IL', 'Illinois'),
 ('Eve', 'FL', 'Unknown')]

In [68]:
@udf(StringType())
def get_full_state_name(state_code):
    return broadcast_states.value.get(state_code, "Unknown")

df.withColumn('FullStateName', get_full_state_name(df['StateCode'])).show()

+-------+---------+-------------+
|   Name|StateCode|FullStateName|
+-------+---------+-------------+
|  Alice|       CA|   California|
|    Bob|       NY|     New York|
|Charlie|       TX|        Texas|
|  David|       IL|     Illinois|
|    Eve|       FL|      Unknown|
+-------+---------+-------------+



In [69]:
broadcast_states.unpersist()